# Brouillon 7 : le médiateur et la causalité inverse

Deux pièges de plus :
1. Médiateur : contrôler une variable qui se trouve *sur le chemin* entre la cause et l'effet fait disparaître
   une partie du vrai effet. Ce n'est pas le collider (vu dans le brouillon 4) : ici la variable est causée par
   la cause, et cause elle-même l'effet.
2. Causalité inverse : quand Y cause X, la régression (et le ML) trouvent un « effet » de X sur Y, sans pouvoir
   dire dans quel sens va la causalité.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from statsmodels.sandbox.regression.gmm import IV2SLS
import doubleml as dml

## 1. Le médiateur
Question : quel est l'effet de la température sur les noyades ? (par exemple pour anticiper un été plus chaud)

La température agit de deux façons :
- directement (fatigue, coups de chaleur dans l'eau) : +0.1 noyade par degré ;
- via la fréquentation des plages : +10 personnes (en centaines) par degré, et +0.02 noyade par centaine de
  personnes, soit +0.2 noyade par degré.

Effet total = 0.1 + 10 × 0.02 = 0.3 ; effet direct = 0.1.

In [2]:
def simuler_mediateur(n=5000, seed=2024):
    rng = np.random.default_rng(seed)
    temperature = rng.normal(20, 6, n)
    frequentation = 50 + 10 * temperature + rng.normal(0, 30, n)
    noyades = 0.1 * temperature + 0.02 * frequentation + rng.normal(0, 2, n)
    return pd.DataFrame({"temperature": temperature, "frequentation": frequentation, "noyades": noyades})

med = simuler_mediateur()
med.corr().round(3)

,temperature,frequentation,noyades
temperature,1.000,0.894,0.648
frequentation,0.894,1.000,0.677
noyades,0.648,0.677,1.000


In [3]:
def ols(df, y, variables, cible):
    m = sm.OLS(df[y], sm.add_constant(df[variables])).fit()
    return m.params[cible], *m.conf_int().loc[cible].tolist()

print("noyades ~ T               :", np.round(ols(med, "noyades", ["temperature"], "temperature"), 4))
print("noyades ~ T + fréquentation :", np.round(ols(med, "noyades", ["temperature", "frequentation"], "temperature"), 4))

noyades ~ T               : [0.2963 0.2866 0.3059]
noyades ~ T + fréquentation : [0.0987 0.0781 0.1193]


- Sans contrôle : 0.3, l'effet total. C'est la bonne réponse à « que se passe-t-il si l'été est plus chaud ? ».
- En contrôlant la fréquentation : 0.1, l'effet direct. On a « coupé » le chemin par la fréquentation.

Aucune des deux n'est fausse : elles répondent à deux questions différentes. L'erreur est de contrôler la
fréquentation par réflexe (« plus de contrôles = mieux ») quand on veut l'effet total.

Est-ce que le Double ML change quelque chose ? Non : il estime l'effet « à contrôles fixés », donc l'effet direct
si on lui donne la fréquentation.

In [4]:
def double_ml(df, y, d, x_cols, seed=0):
    donnees = dml.DoubleMLData(df, y_col=y, d_cols=d, x_cols=x_cols)
    ml_l = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=seed, n_jobs=-1)
    ml_m = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=seed, n_jobs=-1)
    np.random.seed(seed)
    m = dml.DoubleMLPLR(donnees, ml_l, ml_m, n_folds=5).fit()
    return m.coef[0], *m.confint().iloc[0].tolist()

print("Double ML (contrôle = fréquentation) :", np.round(double_ml(med, "noyades", "temperature", ["frequentation"]), 4))

Double ML (contrôle = fréquentation) : [0.099  0.0785 0.1196]


0.1 aussi : l'algorithme ne décide pas à notre place quelle question on pose.

Et le cas d'un médiateur pour les glaces ? Dans notre monde, les glaces n'ont pas d'effet, donc pas de médiateur
intéressant. Je garde l'exemple de la température, plus naturel.

## 2. La causalité inverse
Exemple : le nombre de maîtres-nageurs déployés sur une plage. Les mairies en mettent plus là où il y a des
noyades (les noyades causent les maîtres-nageurs), et les maîtres-nageurs réduisent les noyades
(-0.2 noyade par maître-nageur).

Simultanéité : N = base + b·M + e et M = c·N + u. Je résous le système pour simuler :
N = (base + b·u + e) / (1 - b·c).

J'ajoute un instrument pour plus tard : une dotation de maîtres-nageurs tirée au sort (subvention régionale
attribuée par loterie), qui ajoute 3 maîtres-nageurs sans rapport avec les noyades.

In [5]:
def simuler_causalite_inverse(n=5000, seed=99, effet=-0.2, reaction=1.5):
    rng = np.random.default_rng(seed)
    temperature = rng.normal(20, 6, n)
    dotation = rng.binomial(1, 0.3, n)
    base = 0.3 * temperature + 0.01 * temperature**2 + rng.normal(0, 2, n)
    u = 3 * dotation + rng.normal(0, 1, n)
    noyades = (base + effet * u) / (1 - effet * reaction)
    maitres_nageurs = reaction * noyades + u
    return pd.DataFrame({"temperature": temperature, "dotation": dotation,
                         "maitres_nageurs": maitres_nageurs, "noyades": noyades})

inv = simuler_causalite_inverse()
inv.describe().round(2)

,temperature,dotation,maitres_nageurs,noyades
count,5000.00,5000.00,5000.00,5000.00
mean,20.08,0.30,12.71,7.87
std,6.00,0.46,5.57,3.62
min,-1.42,0.00,-7.46,-5.77
25%,15.96,0.00,8.75,5.33
50%,20.21,0.00,12.52,7.68
75%,24.09,1.00,16.34,10.19
max,40.10,1.00,37.36,24.35


Vérification de la simulation : le résidu e = N - (0.3 T + 0.01 T² - 0.2 M) doit être de variance ≈ 4
(le bruit N(0, 2)) et indépendant de la dotation.

In [6]:
e = inv.noyades - (0.3 * inv.temperature + 0.01 * inv.temperature**2 - 0.2 * inv.maitres_nageurs)
print("variance de e :", round(e.var(), 3), " corr(e, dotation) :", round(np.corrcoef(e, inv.dotation)[0, 1], 4))

variance de e : 4.05  corr(e, dotation) : -0.0215


Variance ≈ 4 (le bruit N(0, 2)) et pas de lien avec la dotation : la simulation est correcte.

In [7]:
inv["temperature2"] = inv.temperature**2
print("OLS naïf                 :", np.round(ols(inv, "noyades", ["maitres_nageurs"], "maitres_nageurs"), 4))
print("OLS + T + T²             :", np.round(ols(inv, "noyades", ["maitres_nageurs", "temperature", "temperature2"], "maitres_nageurs"), 4))
print("Double ML (contrôle = T) :", np.round(double_ml(inv, "noyades", "maitres_nageurs", ["temperature"]), 4))

OLS naïf                 : [0.6187 0.6131 0.6242]
OLS + T + T²             : [0.4603 0.4498 0.4709]


Double ML (contrôle = T) : [0.4557 0.4453 0.4661]


Tout le monde trouve un effet positif : « plus de maîtres-nageurs, plus de noyades ». Contrôler la température
ne sert à rien ici : le problème n'est pas un confondeur, c'est le sens de la causalité.

Le Gradient Boosting aussi ?

In [8]:
gb = GradientBoostingRegressor(random_state=0).fit(inv[["maitres_nageurs"]], inv.noyades)
X1 = inv[["maitres_nageurs"]] + 1
print("effet implicite du GB :", round(np.mean(gb.predict(X1) - gb.predict(inv[["maitres_nageurs"]])), 4))

effet implicite du GB : 0.6179


Oui. Et il prédit très bien : connaître le nombre de maîtres-nageurs renseigne sur les noyades.

La corrélation est symétrique : la régression dans l'autre sens (maîtres-nageurs ~ noyades) donne aussi une pente
positive. Rien dans les données seules ne dit quel est le sens.

In [9]:
print("maîtres-nageurs ~ noyades :", np.round(ols(inv, "maitres_nageurs", ["noyades"], "noyades"), 4))

maîtres-nageurs ~ noyades : [1.4636 1.4504 1.4767]


### Solution : une source de variation qui ne dépend pas des noyades
La dotation tirée au sort joue le rôle d'instrument (ou d'expérience naturelle).

In [10]:
def iv(df, y, d, z, controles=[]):
    X = sm.add_constant(df[[d] + controles])
    Z = sm.add_constant(df[[z] + controles])
    res = IV2SLS(df[y], X, instrument=Z).fit()
    f = sm.OLS(df[d], Z).fit().tvalues[z] ** 2  # F de l'instrument seul (1 instrument : F = t²)
    return res.params[d], *res.conf_int().loc[d].tolist(), f

print("IV (dotation)         :", np.round(iv(inv, "noyades", "maitres_nageurs", "dotation"), 4))
print("IV (dotation) + T, T² :", np.round(iv(inv, "noyades", "maitres_nageurs", "dotation", ["temperature", "temperature2"]), 4))

IV (dotation)         : [-2.709000e-01 -4.135000e-01 -1.283000e-01  1.621804e+02]
IV (dotation) + T, T² : [-2.424000e-01 -3.000000e-01 -1.849000e-01  8.724758e+02]


Les deux sont centrés autour de -0.2, mais sans contrôle l'IC est large : la température ajoute beaucoup de bruit
aux noyades. En la mettant comme variable de contrôle (exogène) dans les deux étapes, l'IC se resserre nettement.
Ici la température n'est pas un confondeur (elle n'affecte pas la dotation) : on la contrôle pour la précision,
comme dans l'expérience aléatoire du brouillon 6.

On retrouve -0.2 : les maîtres-nageurs réduisent les noyades. Le signe s'inverse par rapport à toutes les
méthodes « observationnelles ».

Sensibilité à la force de la réaction (c) :

In [11]:
for c in [0.0, 0.5, 1.5, 3.0]:
    d = simuler_causalite_inverse(reaction=c)
    d["temperature2"] = d.temperature**2
    print(c, "OLS naïf :", round(ols(d, "noyades", ["maitres_nageurs"], "maitres_nageurs")[0], 3),
          " IV + T, T² :", round(iv(d, "noyades", "maitres_nageurs", "dotation", ["temperature", "temperature2"])[0], 3))

0.0 OLS naïf : -0.211  IV + T, T² : -0.231
0.5 OLS naïf : 1.234  IV + T, T² : -0.235
1.5 OLS naïf : 0.619  IV + T, T² : -0.242
3.0 OLS naïf : 0.326  IV + T, T² : -0.255


Sans réaction (c = 0), l'OLS naïf retrouve à peu près -0.2 (la dotation est la seule source de variation des
maîtres-nageurs). Dès que les mairies réagissent aux noyades, le signe s'inverse. Le biais n'est pas monotone en c :
il est maximal pour une réaction modérée, puis diminue quand la réaction devient très forte (les maîtres-nageurs
suivent alors presque exactement les noyades). L'IV reste correct dans tous les cas.

## Bilan pour `src/`
- `simulation.py` : `simuler_mediateur`, `simuler_causalite_inverse`.
- `analyse.py` : `effet_ols` et `effet_double_ml` sont écrits pour glaces/noyades. Plutôt que de tout réécrire,
  j'ajoute un argument optionnel pour le traitement et la variable expliquée (`traitement="glaces"`,
  `cible="noyades"` par défaut), pareil pour `effet_iv` (`instrument="greve"`). Puis `tableau_mediateur` et
  `tableau_causalite_inverse`.
- `graphiques.py` : DAG du médiateur + estimations, nuage maîtres-nageurs / noyades + estimations.